# Learning Analysis Demo

End-to-end walkthrough of the learning-analysis layer in
`src/learning_analysis.py`, combined with the rating and topic models to go
from a raw problem statement to a full learner report: topic/concept/
technique mapping, prerequisites, constraint analysis, difficulty
explanation, readiness ("Can I solve this?"), and a learning path.

## 1. Setup

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

from predict_rating import predict_rating
from predict_topics import predict_topics
from learning_analysis import (
    knowledge_for_tags,
    extract_constraints,
    complexity_advice,
    difficulty_explanation,
    readiness_assessment,
    build_learning_path,
    format_learning_report,
)

sample_statement = """
You are given an array of n integers, where n <= 200000. You need to find
the maximum sum of a contiguous subarray using dynamic programming ideas,
then apply a greedy adjustment to handle at most k allowed removals.
"""

sample_tags = "dp;greedy"

print(sample_statement.strip())

You are given an array of n integers, where n <= 200000. You need to find
the maximum sum of a contiguous subarray using dynamic programming ideas,
then apply a greedy adjustment to handle at most k allowed removals.


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DummyRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator GradientBoostingRegressor from version 1.9.0 when using version 1.8.0. This might lead to

## 2. Three-Level Knowledge Mapping

Maps each Codeforces tag to a Topic, a set of Concepts, and a set of
Techniques.

In [2]:
knowledge = knowledge_for_tags(sample_tags)

print("Topics:", knowledge["topics"])
print()
print("Concepts:")
for item in knowledge["concepts"]:
    print(" -", item)
print()
print("Techniques:")
for item in knowledge["techniques"]:
    print(" -", item)

Topics: ['Dynamic Programming', 'Greedy Algorithms']

Concepts:
 - State definition
 - Transition
 - Base case
 - Local choice
 - Exchange argument

Techniques:
 - Memoization
 - Tabulation
 - State compression
 - Sorting-based greedy
 - Priority queue greedy


## 3. Prerequisite System

For the tags involved, shows what a learner must know, should know, and
what counts as an advanced prerequisite.

In [3]:
print("Must know:")
for item in knowledge["must_know"]:
    print(" -", item)

print()
print("Should know:")
for item in knowledge["should_know"]:
    print(" -", item)

print()
print("Advanced:")
for item in knowledge["advanced"]:
    print(" -", item)

Must know:
 - Arrays
 - Recursion
 - Complexity estimation
 - Sorting

Should know:
 - State transition
 - Overlapping subproblems
 - Proof of correctness
 - Counterexamples

Advanced:
 - Bitmask DP
 - Tree DP
 - DP on graphs
 - Greedy with data structures


## 4. Constraint Analysis

Extracts numeric constraints (like `n <= 200000`) from the statement and
recommends a complexity class, warning against complexities that would be
too slow.

In [4]:
constraints = extract_constraints(sample_statement)
advice = complexity_advice(constraints)

print("Detected constraints:")
for c in constraints:
    print(f" - {c['variable']} <= {c['raw_value']}")

print()
print("Recommended complexity:", advice["recommended"])
print("Avoid:", advice["avoid"])
print("Notes:", advice["notes"])

Detected constraints:
 - n <= 200000

Recommended complexity: ['O(n log n)', 'O(n)']
Avoid: ['O(n^2)']
Notes: ['Look for sorting, greedy, binary search, or data structures.']


## 5. Rating and Topic Prediction

Run the trained models on the sample statement to get a predicted rating
and predicted topics, which feed into the rest of the analysis.

In [5]:
predicted_rating = predict_rating(sample_statement, tags=sample_tags)
predicted_topics = predict_topics(sample_statement)

print("Predicted rating:", predicted_rating)
print()
print("Predicted topics:")
for item in predicted_topics:
    print(f" - {item['topic']} -> {round(item['confidence'] * 100, 1)}%")

Predicted rating: 2008

Predicted topics:
 - data structures -> 54.9%
 - dp -> 53.5%
 - greedy -> 49.2%
 - math -> 45.7%


## 6. Difficulty Explanation

Breaks down why a problem at this rating is difficult: topic contribution,
constraint size, and implementation/proof weight.

In [6]:
explanation = difficulty_explanation(sample_statement, sample_tags, predicted_rating)

print(f"Rating: {explanation['rating']}\n")
print("Difficulty factors:")
for factor in explanation["difficulty_factors"]:
    print(f" - {factor['factor']}: {factor['weight']}%")

Rating: 2008

Difficulty factors:
 - Dynamic Programming: 30%
 - Greedy Algorithms: 30%
 - Large constraints: 10%
 - Advanced implementation/proof: 20%


## 7. Readiness Assessment - "Can I Solve This?"

Given a learner's current rating, solved-problem count, and known topics,
estimate the difficulty gap, success probability, and a verdict.

In [7]:
learner_profile = {
    "user_rating": 1200,
    "solved_count": 150,
    "known_topics": ["Arrays", "Sorting", "Greedy Algorithms"],
}

readiness = readiness_assessment(
    problem_rating=predicted_rating,
    user_rating=learner_profile["user_rating"],
    solved_count=learner_profile["solved_count"],
    known_topics=learner_profile["known_topics"],
    tags=sample_tags,
)

for key, value in readiness.items():
    print(f"{key}: {value}")

problem_rating: 2008
user_rating: 1200
difficulty_gap: 808
topic_coverage: 0.5
success_probability: 28
verdict: Too difficult right now
missing_topics: ['Dynamic Programming']


## 8. Try a Few Learner Profiles

Compare verdicts for learners at different ratings against the same
problem.

In [8]:
profiles = [
    {"user_rating": 900, "solved_count": 20, "known_topics": ["Arrays"]},
    {"user_rating": 1200, "solved_count": 150, "known_topics": ["Arrays", "Sorting", "Greedy Algorithms"]},
    {"user_rating": 1900, "solved_count": 400, "known_topics": ["Arrays", "Sorting", "Greedy Algorithms", "Dynamic Programming"]},
]

for profile in profiles:
    result = readiness_assessment(
        problem_rating=predicted_rating,
        user_rating=profile["user_rating"],
        solved_count=profile["solved_count"],
        known_topics=profile["known_topics"],
        tags=sample_tags,
    )
    print(
        f"Rating {profile['user_rating']:>4} | gap {result['difficulty_gap']:+5d} | "
        f"success {result['success_probability']:>3}% | {result['verdict']}"
    )

Rating  900 | gap +1108 | success   5% | Too difficult right now
Rating 1200 | gap  +808 | success  28% | Too difficult right now
Rating 1900 | gap  +108 | success  75% | Challenging but useful


## 9. Learning Path

In [9]:
path = build_learning_path(
    current_rating=learner_profile["user_rating"],
    target_rating=predicted_rating,
    known_topics=learner_profile["known_topics"],
    tags=sample_tags,
)

print(f"Path from {learner_profile['user_rating']} to {predicted_rating}:\n")
for step_number, step in enumerate(path, start=1):
    print(f"{step_number}. {step}")

Path from 1200 to 2008:

1. Binary search and two pointers
2. Basic dynamic programming
3. Graphs, DFS, BFS, and stronger DP
4. Advanced DP and data structures
5. Dynamic Programming


## 10. Full Report

Puts everything together into a single formatted report, the same one used
by `src/learning_assistant.py --analyze` and the Streamlit "Learning
Analysis" / "Readiness Check" tabs.

In [10]:
report = format_learning_report(
    statement=sample_statement,
    tags=sample_tags,
    problem_rating=predicted_rating,
    user_rating=learner_profile["user_rating"],
    solved_count=learner_profile["solved_count"],
    known_topics=learner_profile["known_topics"],
)

print(report)

Learning analysis
Problem rating: 2008

Topics you should know:
- Dynamic Programming
- Greedy Algorithms

Important concepts:
- State definition
- Transition
- Base case
- Local choice
- Exchange argument

Techniques:
- Memoization
- Tabulation
- State compression
- Sorting-based greedy
- Priority queue greedy

Prerequisites - must know:
- Arrays
- Recursion
- Complexity estimation
- Sorting

Prerequisites - should know:
- State transition
- Overlapping subproblems
- Proof of correctness
- Counterexamples

Advanced prerequisites:
- Bitmask DP
- Tree DP
- DP on graphs
- Greedy with data structures

Constraint analysis:
- n <= 200000
Recommended complexity:
- O(n log n)
- O(n)
Avoid:
- O(n^2)

Why this problem is difficult:
- Dynamic Programming: 30%
- Greedy Algorithms: 30%
- Large constraints: 10%
- Advanced implementation/proof: 20%

Can I solve this?
- Your rating: 1200
- Difficulty gap: +808
- Estimated success: 28%
- Verdict: Too difficult right now
- Missing topics: Dynamic Progr